# Soul Tower Analytics Notebook

This notebook is your exploration workspace for Soul Tower's game data. Every cell is runnable and annotated, so you can learn the analytics system by walking top to bottom.

**What you can do here:**
- Load every spreadsheet into a DataFrame
- Search for any ability across every sheet at once
- Convert die-based mana costs to numeric bounds
- Group cards by type, subtype, or origin
- Merge static card data with live Flask analytics

**Prerequisites:**
- You've run `python main.py --fresh` at least once to populate `data/cache/`
- If you want live analytics, the Flask server must be running on `localhost:5050`

**How to extend this notebook:**
- New analytics questions become new cells, not new files
- When a cell becomes reusable, move it to `src/analytics/query.py` as a function
- Anything that returns a DataFrame can be chained with pandas methods — `.groupby()`, `.query()`, `.merge()`

## 1. Setup

Make `query.py` importable. The `sys.path.insert` hack lets you run this notebook from anywhere in the project.

In [ ]:
import sys
from pathlib import Path

# Add the project root to sys.path so we can import src.analytics.query
project_root = Path.cwd()
while project_root.name and not (project_root / 'src' / 'analytics' / 'query.py').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.analytics import query
import pandas as pd

# Show all columns in DataFrames (pandas truncates by default)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)

print(f'Project root: {project_root}')
query.summary()

## 2. Load everything

One call gives you a dict of five DataFrames keyed by sheet name. Store them in a variable and use as needed.

In [ ]:
sheets = query.load_all()

heroes   = sheets['hero']
commons  = sheets['common_cards']
legends  = sheets['legendary']
calams   = sheets['calamity']
villains = sheets['villain']

print(f'Heroes:         {len(heroes)}')
print(f'Common cards:   {len(commons)}')
print(f'Legendary:      {len(legends)}')
print(f'Calamities:     {len(calams)}')
print(f'Villains:       {len(villains)}')

## 3. Peek at one sheet

`.head()` shows the first few rows. `.columns` shows you what's available for querying.

In [ ]:
heroes.head()

In [ ]:
# See every column name in the Hero sheet
list(heroes.columns)

## 4. Find every card and hero that uses an ability

`query.find_ability()` is the most powerful exploration tool. It searches every effect and passive column across every sheet for a keyword and returns a unified DataFrame showing exactly where each match was found.

Try changing `'Enchant'` to any ability you care about: `'Attune'`, `'Forsake'`, `'Deal'`, `'Blood Price'`.

In [ ]:
query.find_ability('Enchant')

In [ ]:
# Filter to just one source
enchants = query.find_ability('Enchant')
enchants[enchants['source'] == 'legendary']

In [ ]:
# Or group by source to see counts
query.find_ability('Enchant').groupby('source').size()

## 5. Card cost analysis

Card costs in the spreadsheet are strings. Some are plain numbers (`'3'`), some are die expressions (`'2d4'`). `query.add_cost_columns()` parses them all and gives you three numeric columns: `cost_min`, `cost_max`, `cost_avg`.

Once those are in place, all the usual pandas math works.

In [ ]:
commons_costed = query.add_cost_columns(commons)
commons_costed[['Name', 'Type', 'Subtype', 'Cost', 'cost_min', 'cost_max', 'cost_avg', 'cost_is_die']].head(10)

In [ ]:
# Average cost per card type
commons_costed.groupby('Type')['cost_avg'].agg(['mean', 'min', 'max', 'count'])

In [ ]:
# Which cards use die-based costs?
commons_costed[commons_costed['cost_is_die']][['Name', 'Type', 'Subtype', 'Cost', 'cost_min', 'cost_max']]

## 6. All playable cards in one table

Common cards and Legendary cards live in separate sheets but share most columns. `query.cards_by_type()` unions them with a `card_source` column so you can treat them as one dataset.

In [ ]:
all_cards = query.cards_by_type()
print(f'Total cards: {len(all_cards)}')

# Count by source and type
all_cards.groupby(['card_source', 'Type']).size().unstack(fill_value=0)

## 7. Group by Origin

For Heroes, this shows stat averages per Origin — useful for checking design rules like the Imanis 5 Health minimum.

For card sheets, it shows card count and cost ranges.

In [ ]:
query.by_origin('hero')

In [ ]:
# Check: do all Imanis heroes have Health >= 5?
imanis = heroes[heroes['Origin'] == 'Imanis']
imanis[['Nickname', 'Health', 'Might', 'Speed', 'Luck', 'Arcana']]

In [ ]:
# Flag any violations
violations = heroes[(heroes['Origin'] == 'Imanis') & (heroes['Health'].astype(int) < 5)]
if len(violations) == 0:
    print('✓ All Imanis heroes meet the 5 Health minimum')
else:
    print(f'✗ {len(violations)} violations:')
    print(violations[['Nickname', 'Origin', 'Health']])

## 8. Find cards a specific Hero connects to

Every Legendary card has an Origin field pointing to its Hero. This lets you look up Akiem's signature cards without scanning by hand.

In [ ]:
def hero_cards(hero_nickname):
    '''Return all Legendary cards that belong to a given Hero.'''
    return legends[legends['Origin'] == hero_nickname][['Name', 'Type', 'Subtype', 'Cost']]

# Change 'Akiem' to any Hero nickname
hero_cards('Akiem')

## 9. Live analytics hook

If your Flask analytics server is running, you can merge live playtest data into the static card data. If it's not running, these calls return `None` silently — no exceptions.

Requires: `python backend/analytics_server.py` running in another terminal.

In [ ]:
sessions = query.live_sessions()
if sessions is None:
    print('Flask server not reachable at localhost:5050 (this is fine, skip this section)')
else:
    print(f'Found {len(sessions)} playtest sessions')
    sessions.head()

In [ ]:
# Merge hero data with live pick/defeat stats
combined = query.with_live_analytics('hero')
combined.head()

## 10. Your scratchpad

Everything below is yours. Ask questions, build plots, explore patterns. When a question becomes something you'll ask again, promote it to `query.py` as a function.

In [ ]:
# Example: which cards have the highest average cost?
all_cards.nlargest(10, 'cost_avg')[['Name', 'Type', 'Subtype', 'Cost', 'cost_avg', 'card_source']]

In [ ]:
# Example: regex search flavor text for recurring themes
import re

def search_flavor(pattern, df=heroes, col='Flavor Text'):
    if col not in df.columns:
        return pd.DataFrame()
    mask = df[col].astype(str).str.contains(pattern, case=False, regex=True, na=False)
    return df[mask][['Nickname' if 'Nickname' in df.columns else 'Name', col]]

# Change the pattern to whatever theme you want to explore
# search_flavor(r'\b(blood|curse|shadow)\b')